In [47]:
import pandas as pd
import os
import numpy as np
import re
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, f1_score
from scipy.sparse import csr_matrix, load_npz, vstack
import pickle

from sklearn.model_selection import train_test_split

In [42]:
import kagglehub

In [67]:
path = kagglehub.dataset_download("starblasters8/human-vs-llm-text-corpus")

print("Path to dataset files:", path)
os.listdir(path)

Path to dataset files: C:\Users\osten\.cache\kagglehub\datasets\starblasters8\human-vs-llm-text-corpus\versions\2


['data.csv',
 'data.parquet',
 'distribution.csv',
 'distribution.parquet',
 'prompts.csv',
 'prompts.parquet']

In [68]:
df_llm = pd.read_csv(os.path.join(path,'data.csv'))

In [69]:
df_lower = df_llm.copy()

#Lowercasing
for col in df_llm.select_dtypes(include=['object']).columns:
        df_lower[col] = df_lower[col].str.lower()

#Noise Removal
def remove_noise(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z0-9.\s]', '', text)
    return text

df_lower['text'] = df_lower['text'].apply(remove_noise)

# Tokenization
df_lower['tokenized_text'] = df_lower['text'].apply(lambda x: x.split())

#Filtering
min_text_length = 30
df_filtered = df_lower[df_lower['tokenized_text'].apply(len) >= min_text_length].copy()

df_filtered

,text,source,prompt_id,text_length,word_count,tokenized_text
0,federal law supersedes state law and cannabis ...,bloom-7b,0,967,157,"[federal, law, supersedes, state, law, and, ca..."
1,miles feels restless after working all day. he...,bloom-7b,0,5068,778,"[miles, feels, restless, after, working, all, ..."
2,so first of i am danish. that means that i fol...,bloom-7b,0,1602,267,"[so, first, of, i, am, danish., that, means, t..."
3,in this paper we present a novel rulebased app...,bloom-7b,0,5469,848,"[in, this, paper, we, present, a, novel, ruleb..."
4,most social progressives love democracy and so...,bloom-7b,0,2379,380,"[most, social, progressives, love, democracy, ..."
...,...,...,...,...,...,...
788917,\nin the vast expanse of time where the echoes...,yi-34b,1293,5523,870,"[in, the, vast, expanse, of, time, where, the,..."
788918,\nthe phenomenon of brain drain particularly i...,yi-34b,1154,4540,677,"[the, phenomenon, of, brain, drain, particular..."
788919,\nthe influence of climate change on marine ec...,yi-34b,2783,3889,598,"[the, influence, of, climate, change, on, mari..."
788920,\ntitle the case for limiting car usage naviga...,yi-34b,41,3560,533,"[title, the, case, for, limiting, car, usage, ..."


In [75]:
df_filtered['generated']=df_filtered['source'].map(lambda x: 0 if x =='human' else 1)

In [85]:
df_filtered['generated'].value_counts()

generated
1    439145
0    343730
Name: count, dtype: int64

In [50]:
df_filtered = df_filtered.rename(columns={'source':'generated'})

In [23]:
df_preprocessed = pd.read_csv('downsampled.csv')

In [103]:
df_ai = df_filtered[df_filtered['generated']==1]
df_human = df_filtered[df_filtered['generated']==0]

In [106]:
df_ai = df_ai.sample(n=len(df_human), random_state=3)

In [107]:
df_ai

,text,source,prompt_id,text_length,word_count,tokenized_text,generated
658030,full disclosure nearly just got killed by a se...,opt-30b,0,406,78,"[full, disclosure, nearly, just, got, killed, ...",1
737885,one could argue that insisting your so does no...,text-davinci-002,0,317,55,"[one, could, argue, that, insisting, your, so,...",1
33570,this was formerly risque which was the worst v...,flan-t5-base,0,216,45,"[this, was, formerly, risque, which, was, the,...",1
699598,technology has become an integral part of our...,palm-2,1328,2276,375,"[technology, has, become, an, integral, part, ...",1
683912,\n\nthe advent of television has undeniably re...,openchat-3.5,1044,3703,536,"[the, advent, of, television, has, undeniably,...",1
...,...,...,...,...,...,...,...
656659,im in shopping heaven usually im a little skep...,opt-30b,0,613,109,"[im, in, shopping, heaven, usually, im, a, lit...",1
16633,the sun peeked through the blinds in my room ...,claude-v1,1,2530,430,"[the, sun, peeked, through, the, blinds, in, m...",1
12281,dear senator\n\ni am writing to express my vie...,claude-instant-v1,2,2233,340,"[dear, senator, i, am, writing, to, express, m...",1
641544,the amazon rainforest portuguese floresta amaz...,opt-2.7b,0,3120,481,"[the, amazon, rainforest, portuguese, floresta...",1


In [108]:
df_filtered1=pd.concat([df_human,df_ai],ignore_index=True)

In [82]:
df_filtered1['source'].value_counts()

source
human    687460
Name: count, dtype: int64

In [110]:
df_filtered1['generated'].value_counts()

generated
0    343730
1    343730
Name: count, dtype: int64

In [111]:
train_df, test_df = train_test_split(df_filtered1, test_size=0.2, random_state=3, stratify=df_filtered1['generated'])

In [25]:
#Entropy
def calculate_entropy(text_tokens):
    if not text_tokens:
        return 0.0
    counts = Counter(text_tokens)
    probabilities = [count / len(text_tokens) for count in counts.values()]
    entropy = -sum(p * math.log2(p) for p in probabilities)
    return entropy

train_df['entropy'] = train_df['tokenized_text'].apply(calculate_entropy)
test_df['entropy'] = test_df['tokenized_text'].apply(calculate_entropy)
df_preprocessed['entropy'] = df_preprocessed['tokenized_text'].apply(calculate_entropy)

In [26]:
# Burstiness
def calculate_burstiness(text):
    if not isinstance(text, str):
        return 0.0 # Handle non-string input
    sentences = sent_tokenize(text)
    if len(sentences) < 2:
        return 0.0 # Need at least 2 sentences to calculate variance
    sentence_lengths = [len(s.split()) for s in sentences]
    # Avoid division by zero if all sentence lengths are the same
    if len(set(sentence_lengths)) == 1:
        return 0.0
    return pd.Series(sentence_lengths).var()

train_df['burstiness'] = train_df['text'].apply(calculate_burstiness)
test_df['burstiness'] = test_df['text'].apply(calculate_burstiness)
df_preprocessed['burstiness'] = df_preprocessed['text'].apply(calculate_burstiness)

In [27]:
# Lexical Diversity
def calculate_lexical_diversity(text_tokens):
    if not text_tokens:
        return 0.0
    return len(set(text_tokens)) / len(text_tokens)

train_df['lexical_diversity'] = train_df['tokenized_text'].apply(calculate_lexical_diversity)
test_df['lexical_diversity'] = test_df['tokenized_text'].apply(calculate_lexical_diversity)
df_preprocessed['lexical_diversity'] = df_preprocessed['tokenized_text'].apply(calculate_lexical_diversity)

In [28]:
# TF-IDF Vector
vectorizer = TfidfVectorizer(max_features=5000)
tfidf_train = vectorizer.fit_transform(train_df['text'])
tfidf_test = vectorizer.transform(test_df['text'])
tfidf = vectorizer.transform(df_preprocessed['text'])

In [29]:
y_train = train_df['generated'].values
y_test = test_df['generated'].values

In [30]:
stylo_cols = ['entropy', 'burstiness', 'lexical_diversity']
stylo_train = csr_matrix(train_df[stylo_cols].values)
stylo_test = csr_matrix(test_df[stylo_cols].values)
stylo_prep = csr_matrix(df_preprocessed[stylo_cols].values)

feature_configs = {
    'tfidf': (tfidf_train, tfidf_test),
    'stylo': (stylo_train, stylo_test),
    'tfidf_ent': (hstack([tfidf_train, csr_matrix(train_df[['entropy']].values)]),
                  hstack([tfidf_test, csr_matrix(test_df[['entropy']].values)])),
    'tfidf_burst': (hstack([tfidf_train, csr_matrix(train_df[['burstiness']].values)]),
                    hstack([tfidf_test, csr_matrix(test_df[['burstiness']].values)])),
    'tfidf_lex': (hstack([tfidf_train, csr_matrix(train_df[['lexical_diversity']].values)]),
                  hstack([tfidf_test, csr_matrix(test_df[['lexical_diversity']].values)])),
    'tfidf_stylo': (hstack([tfidf_train, stylo_train]),
                    hstack([tfidf_test, stylo_test]))
}

In [31]:
results = []

In [32]:
for config_name, (X_train, X_test) in feature_configs.items():
    print(f"\n{'='*60}\n{config_name.upper()}\n{'='*60}")
    
    fold_metrics = {'train_acc': [], 'train_auc': [], 'train_f1': [],
                    'val_acc': [], 'val_auc': [], 'val_f1': []}
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        model = LinearSVC(C=5, max_iter=10000, random_state=0)
        model.fit(X_tr, y_tr)
        
        tr_pred = model.predict(X_tr)
        tr_prob = model.decision_function(X_tr)
        val_pred = model.predict(X_val)
        val_prob = model.decision_function(X_val)
        
        fold_metrics['train_acc'].append(accuracy_score(y_tr, tr_pred))
        fold_metrics['train_auc'].append(roc_auc_score(y_tr, tr_prob))
        fold_metrics['train_f1'].append(f1_score(y_tr, tr_pred))
        fold_metrics['val_acc'].append(accuracy_score(y_val, val_pred))
        fold_metrics['val_auc'].append(roc_auc_score(y_val, val_prob))
        fold_metrics['val_f1'].append(f1_score(y_val, val_pred))
        
        print(f"F{fold} Train: Acc={fold_metrics['train_acc'][-1]:.3f} "
              f"AUC={fold_metrics['train_auc'][-1]:.3f} F1={fold_metrics['train_f1'][-1]:.3f}")
        print(f"F{fold} Val:   Acc={fold_metrics['val_acc'][-1]:.3f} "
              f"AUC={fold_metrics['val_auc'][-1]:.3f} F1={fold_metrics['val_f1'][-1]:.3f}")
    
    avg_metrics = {k: np.mean(v) for k, v in fold_metrics.items()}
    std_metrics = {k: np.std(v) for k, v in fold_metrics.items()}
    
    print(f"\nMean ± Std:")
    print(f"Train: Acc={avg_metrics['train_acc']:.3f}±{std_metrics['train_acc']:.3f} "
          f"AUC={avg_metrics['train_auc']:.3f}±{std_metrics['train_auc']:.3f} "
          f"F1={avg_metrics['train_f1']:.3f}±{std_metrics['train_f1']:.3f}")
    print(f"Val:   Acc={avg_metrics['val_acc']:.3f}±{std_metrics['val_acc']:.3f} "
          f"AUC={avg_metrics['val_auc']:.3f}±{std_metrics['val_auc']:.3f} "
          f"F1={avg_metrics['val_f1']:.3f}±{std_metrics['val_f1']:.3f}")
    
    model_final = LinearSVC(C=5, max_iter=10000, random_state=0)
    model_final.fit(X_train, y_train)
    
    test_pred = model_final.predict(X_test)
    test_prob = model_final.decision_function(X_test)
    
    test_acc = accuracy_score(y_test, test_pred)
    test_auc = roc_auc_score(y_test, test_prob)
    test_f1 = f1_score(y_test, test_pred)
    
    print(f"\nHoldout Test: Acc={test_acc:.3f} AUC={test_auc:.3f} F1={test_f1:.3f}")
    
    results.append({
        'config': config_name,
        **{f'{k}_mean': v for k, v in avg_metrics.items()},
        **{f'{k}_std': v for k, v in std_metrics.items()},
        'test_acc': test_acc,
        'test_auc': test_auc,
        'test_f1': test_f1
    })


TFIDF
F1 Train: Acc=0.999 AUC=1.000 F1=0.999
F1 Val:   Acc=0.998 AUC=1.000 F1=0.998
F2 Train: Acc=0.999 AUC=1.000 F1=0.999
F2 Val:   Acc=0.998 AUC=1.000 F1=0.998
F3 Train: Acc=0.999 AUC=1.000 F1=0.999
F3 Val:   Acc=0.998 AUC=1.000 F1=0.998
F4 Train: Acc=0.999 AUC=1.000 F1=0.999
F4 Val:   Acc=0.998 AUC=1.000 F1=0.998
F5 Train: Acc=0.999 AUC=1.000 F1=0.999
F5 Val:   Acc=0.997 AUC=1.000 F1=0.997

Mean ± Std:
Train: Acc=0.999±0.000 AUC=1.000±0.000 F1=0.999±0.000
Val:   Acc=0.998±0.000 AUC=1.000±0.000 F1=0.998±0.000

Holdout Test: Acc=0.998 AUC=1.000 F1=0.998

STYLO
F1 Train: Acc=0.729 AUC=0.804 F1=0.725
F1 Val:   Acc=0.729 AUC=0.801 F1=0.725
F2 Train: Acc=0.729 AUC=0.803 F1=0.725
F2 Val:   Acc=0.729 AUC=0.803 F1=0.726
F3 Train: Acc=0.729 AUC=0.803 F1=0.725
F3 Val:   Acc=0.730 AUC=0.805 F1=0.726
F4 Train: Acc=0.727 AUC=0.801 F1=0.724
F4 Val:   Acc=0.725 AUC=0.801 F1=0.722
F5 Train: Acc=0.730 AUC=0.804 F1=0.726
F5 Val:   Acc=0.730 AUC=0.805 F1=0.725

Mean ± Std:
Train: Acc=0.729±0.001 AUC=0

In [33]:
results_df = pd.DataFrame(results)
print(f"\n{'='*60}\nABLATION SUMMARY\n{'='*60}")
print(results_df[['config', 'val_acc_mean', 'val_auc_mean', 'val_f1_mean', 
                  'test_acc', 'test_auc', 'test_f1']].to_string(index=False))



ABLATION SUMMARY
     config  val_acc_mean  val_auc_mean  val_f1_mean  test_acc  test_auc  test_f1
      tfidf      0.997678      0.999945     0.997677  0.998084  0.999960 0.998083
      stylo      0.728645      0.802945     0.724832  0.728917  0.801751 0.724226
  tfidf_ent      0.997681      0.999945     0.997680  0.998029  0.999962 0.998028
tfidf_burst      0.992289      0.999296     0.992275  0.993412  0.999521 0.993395
  tfidf_lex      0.997829      0.999949     0.997829  0.998139  0.999963 0.998138
tfidf_stylo      0.992961      0.999383     0.992948  0.994239  0.999665 0.994227
